# Transformer Encoder Classifier -- Majority-Token Task (Evaluation)

Loads the checkpoint produced by
`notebooks/colab/transformers/majority_token_train.ipynb` and runs
everything that does NOT require retraining:

1. **Test accuracy + confusion matrix** over the full held-out split.
2. **Loss/accuracy curves** (re-plotted from the saved JSON-lines log).
3. **Sample predictions**: a handful of test sequences shown as
   colored token strips, titled with true vs. predicted majority
   value.
4. **CLS attention maps**: the last encoder block's attention weights
   from the `[CLS]` token to every sequence position -- the
   Transformer-specific diagnostic. If the model has learned the task,
   attention mass should concentrate on the positions holding the true
   majority value (highlighted in red).

All evaluation/visualization logic lives in
`code/transformers/majority_token_evaluate.py`; this notebook only
calls it and displays the results.

## 1. GPU check

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code", "transformers"))


## 3. Mount Drive, load the checkpoint

`CHECKPOINT_PATH`/`LOG_PATH` must match the paths used in the training
notebook. The model is reconstructed from the config saved INSIDE the
checkpoint -- no architecture hyperparameter needs to be retyped here.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/thesis_data/majority_token/checkpoint.pt"
LOG_PATH = "/content/drive/MyDrive/thesis_data/majority_token/train_log.jsonl"

from majority_token_evaluate import load_model_and_config

model, config = load_model_and_config(CHECKPOINT_PATH, device)
print(f"loaded checkpoint from {CHECKPOINT_PATH}")
print(f"config = {config}")


## 4. Rebuild the test split and compute test accuracy + confusion matrix

In [ ]:
from majority_token_dataset import make_majority_token_datasets
from majority_token_evaluate import compute_test_accuracy_and_confusion, plot_confusion_matrix
import matplotlib.pyplot as plt

_, test_dataset = make_majority_token_datasets(
    config["vocab_size"], config["seq_len"], config["num_train"], config["num_test"], seed=config["seed"]
)

accuracy, confusion = compute_test_accuracy_and_confusion(model, test_dataset, device)
print(f"test accuracy = {accuracy:.4f}  (chance level = {1 / config['vocab_size']:.4f})")

plot_confusion_matrix(confusion)
plt.show()


## 5. Loss / accuracy curves

In [ ]:
from majority_token_evaluate import plot_training_curves

plot_training_curves(LOG_PATH)
plt.show()


## 6. Sample predictions (input sequence, true vs. predicted majority token)

In [ ]:
from majority_token_evaluate import visualize_predictions

visualize_predictions(model, test_dataset, device, num_examples=5)
plt.show()


## 7. CLS attention maps

Last encoder block's attention weights from `[CLS]` (query = position
0) to every position, averaged over heads. Red bars mark positions
that hold the true majority value -- a well-trained model's CLS token
should attend to the red bars more than the gray ones.

In [ ]:
from majority_token_evaluate import visualize_cls_attention

visualize_cls_attention(model, test_dataset, device, num_examples=4, layer=-1)
plt.show()


## 8. Next step

This exercise used a generic token-sequence task so the Transformer
encoder itself (multi-head attention, residual connections, LayerNorm,
MLP) could be exercised and visualized directly. A later exercise
reuses the same `TransformerEncoderBlock` with a patch-embedding front
end (image -> sequence of patch tokens) to build a Vision Transformer
(ViT).